In [ ]:
from pyspark.sql import functions as F
from pyspark.sql import types as T
import boto3, json, botocore

# -----------------------------
# CREDENTIALS RETRIEVAL (AWS SM)
# -----------------------------
SERVICE_CREDENTIAL_NAME = "asset_api_nprd_rds_service_credential"
REGION = "eu-west-1"

provider = dbutils.credentials.getServiceCredentialsProvider(SERVICE_CREDENTIAL_NAME)
session = boto3.Session(botocore_session=provider, region_name=REGION)
sm = session.client("secretsmanager")

response = sm.get_secret_value(SecretId="demo-document-bucket")
creds = json.loads(response["SecretString"])

DB_HOST = creds["DB_HOST"]
DB_NAME = creds["DB_NAME"]
DB_USER = creds["DB_USER"]
DB_PASSWORD = creds["DB_PASSWORD"]
DB_PORT = creds["DB_PORT"]

DB_URL = f"jdbc:postgresql://{DB_HOST}:{DB_PORT}/{DB_NAME}"
DB_PROPS = {
  "user": DB_USER,
  "password": DB_PASSWORD,
  "driver": "org.postgresql.Driver",
  "fetchsize": "10000"
}

# =========================
# CONFIG DELTA (Unity Catalog)
# =========================
CATALOG = "asset_api_webapp_dev"
SCHEMA  = "default"

WATERMARK_TABLE = f"{CATALOG}.{SCHEMA}._rds_mirror_watermarks_v3_fk_source_ids_noprod"

TABLES = [
  {
    "pg_table": "public.kpi_log",
    "delta_table": f"{CATALOG}.{SCHEMA}.kpi_log_delta_noprod",
    "pk_col": "id",
    "created_col": "changed_at",
    "updated_col": "changed_at",
    "fk_mappings": [],   
  }, 
  {
    "pg_table": "public.kpi_data",
    "delta_table": f"{CATALOG}.{SCHEMA}.kpi_data_delta_noprod",
    "pk_col": "id",
    "created_col": "created_date",
    "updated_col": "updated_date",
    "fk_mappings": [
      ("signal_id",  "public.signal",  "id", "source_id", "signal_source_id"),
      ("unit_id",    "public.unit",    "id", "source_id", "unit_source_id"),
      ("version_id", "public.version", "id", "source_id", "version_source_id"),
    ],
  },
  {
    "pg_table": "public.kpi_comment",
    "delta_table": f"{CATALOG}.{SCHEMA}.kpi_comment_delta_noprod",
    "pk_col": "id",
    "created_col": "created_date",
    "updated_col": "updated_date",
    "fk_mappings": [
      ("signal_id",  "public.signal", "id", "source_id", "signal_source_id"),
      ("unit_id",    "public.unit",   "id", "source_id", "unit_source_id"),
    ],
  },
]

# -----------------------------
# Helpers
# -----------------------------
def qname(full_name: str) -> str:
    return ".".join([f"`{p}`" for p in full_name.split(".")])

def df_is_empty(df) -> bool:
    return len(df.take(1)) == 0

def read_lookup(table: str, pk: str, source_col: str):
    q = f"(SELECT {pk} AS lk_id, {source_col} AS lk_source_id FROM {table}) AS q"
    return spark.read.jdbc(DB_URL, q, properties=DB_PROPS)

def read_user_lookup():
    q = """
    (
      SELECT
        id        AS lk_user_id,
        gid       AS lk_user_gid,
        firstname AS lk_user_firstname,
        lastname  AS lk_user_lastname
      FROM public."user"
    ) AS q
    """
    return spark.read.jdbc(DB_URL, q, properties=DB_PROPS)

def read_exercise_enrich_lookup():
    q = """
    (
      SELECT
        e.id AS lk_exercise_id,
        e.source_id AS lk_exercise_source_id,
        sc.source_id AS lk_scenario_id,
        dd.source_id AS lk_data_domain_source_id
      FROM public.exercise e
      LEFT JOIN public.scenario sc ON e.scenario_id = sc.id
      LEFT JOIN public.data_domain_version ddv ON e.data_domain_version_id = ddv.id
      LEFT JOIN public.data_domain dd ON ddv.data_domain_id = dd.id
    ) AS q
    """
    return spark.read.jdbc(DB_URL, q, properties=DB_PROPS)

# =========================
# WATERMARK TABLE
# =========================
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {qname(WATERMARK_TABLE)} (
  source_table STRING,
  watermark_ts TIMESTAMP,
  watermark_pk STRING
)
USING delta
""")

def get_watermark(source_table: str):
    rows = (spark.table(WATERMARK_TABLE)
            .where(F.col("source_table") == source_table)
            .select("watermark_ts", "watermark_pk")
            .take(1))
    if len(rows) == 0:
        return ("1970-01-01 00:00:00", None)
    r = rows[0]
    return (r["watermark_ts"].strftime("%Y-%m-%d %H:%M:%S"), r["watermark_pk"])

def set_watermark(source_table: str, new_ts_str: str, new_pk):
    new_pk_sql = "NULL" if new_pk is None else f"'{new_pk}'"
    spark.sql(f"""
    MERGE INTO {qname(WATERMARK_TABLE)} t
    USING (
      SELECT
        '{source_table}' AS source_table,
        TIMESTAMP('{new_ts_str}') AS watermark_ts,
        {new_pk_sql} AS watermark_pk
    ) s
    ON t.source_table = s.source_table
    WHEN MATCHED THEN UPDATE SET watermark_ts = s.watermark_ts, watermark_pk = s.watermark_pk
    WHEN NOT MATCHED THEN INSERT (source_table, watermark_ts, watermark_pk)
    VALUES (s.source_table, s.watermark_ts, s.watermark_pk)
    """)

# =========================
# Delta schema helpers
# =========================
def ensure_delta_table_exists_sql(delta_table: str, df_schema_source):
    tmp_view = "__tmp_schema_view__"
    df_schema_source.limit(0).createOrReplaceTempView(tmp_view)
    spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {qname(delta_table)}
    USING delta
    AS SELECT * FROM {tmp_view}
    """)
    spark.catalog.dropTempView(tmp_view)

def spark_type_to_sql(dt: T.DataType) -> str:
    if isinstance(dt, T.StringType): return "STRING"
    if isinstance(dt, T.IntegerType): return "INT"
    if isinstance(dt, T.LongType): return "BIGINT"
    if isinstance(dt, T.DoubleType): return "DOUBLE"
    if isinstance(dt, T.FloatType): return "FLOAT"
    if isinstance(dt, T.BooleanType): return "BOOLEAN"
    if isinstance(dt, T.TimestampType): return "TIMESTAMP"
    if isinstance(dt, T.DateType): return "DATE"
    if isinstance(dt, T.DecimalType): return f"DECIMAL({dt.precision},{dt.scale})"
    return dt.simpleString().upper()

def ensure_delta_schema_has_columns(delta_table: str, src_df):
    tgt_cols = set(spark.table(delta_table).columns)
    missing_fields = [f for f in src_df.schema.fields if f.name not in tgt_cols]
    if not missing_fields:
        return
    add_cols_sql = ", ".join([f"`{f.name}` {spark_type_to_sql(f.dataType)}" for f in missing_fields])
    print(f"Altering {delta_table}: adding columns -> {[f.name for f in missing_fields]}")
    spark.sql(f"ALTER TABLE {qname(delta_table)} ADD COLUMNS ({add_cols_sql})")

def merge_into(delta_table: str, src_df, pk_col: str):
    src_df.createOrReplaceTempView("src_changes")
    cols = src_df.columns
    update_cols = [c for c in cols if c != pk_col]
    set_clause = ",\n      ".join([f"t.`{c}` = s.`{c}`" for c in update_cols])
    insert_cols = ", ".join([f"`{c}`" for c in cols])
    insert_vals = ", ".join([f"s.`{c}`" for c in cols])

    spark.sql(f"""
    MERGE INTO {qname(delta_table)} t
    USING src_changes s
    ON t.`{pk_col}` = s.`{pk_col}`
    WHEN MATCHED THEN UPDATE SET
      {set_clause}
    WHEN NOT MATCHED THEN INSERT ({insert_cols})
    VALUES ({insert_vals})
    """)


def read_kpi_data_enrich_lookup():
    q = """
    (
      SELECT
        kd.id AS lk_kpi_data_id,

        -- ✅ champs demandés depuis kpi_data
        kd.granularity  AS lk_granularity,
        kd.period_year  AS lk_period_year,
        kd.period_month AS lk_period_month,

        e.source_id  AS lk_exercise_source_id,
        sc.source_id AS lk_exercise_scenario_id,
        dd.source_id AS lk_data_domain_source_id,

        sig.source_id AS lk_signal_source_id,
        u.source_id   AS lk_unit_source_id,
        v.source_id   AS lk_version_source_id

      FROM public.kpi_data kd
      LEFT JOIN public.exercise e
        ON kd.exercise_id = e.id
      LEFT JOIN public.scenario sc
        ON e.scenario_id = sc.id
      LEFT JOIN public.data_domain_version ddv
        ON e.data_domain_version_id = ddv.id
      LEFT JOIN public.data_domain dd
        ON ddv.data_domain_id = dd.id

      LEFT JOIN public.signal sig
        ON kd.signal_id = sig.id
      LEFT JOIN public.unit u
        ON kd.unit_id = u.id
      LEFT JOIN public.version v
        ON kd.version_id = v.id
    ) AS q
    """
    return spark.read.jdbc(DB_URL, q, properties=DB_PROPS)

def read_kpi_comment_latest_lookup():
    # 1 ligne par kpi_data_id (dernier comment selon updated_date/created_date)
    q = """
    (
      SELECT
        kc.kpi_data_id AS lk_kpi_data_id,
        kc.comment     AS lk_comment
      FROM public.kpi_comment kc
      JOIN (
        SELECT
          kpi_data_id,
          MAX(COALESCE(updated_date, created_date)) AS max_dt
        FROM public.kpi_comment
        GROUP BY kpi_data_id
      ) last
        ON last.kpi_data_id = kc.kpi_data_id
       AND COALESCE(kc.updated_date, kc.created_date) = last.max_dt
    ) AS q
    """
    return spark.read.jdbc(DB_URL, q, properties=DB_PROPS)

# =========================
# MIRROR GUARDRAIL (IDs only)
# =========================
def should_full_refresh(pg_table: str, delta_table: str, pk_col: str) -> bool:
    """
    FULL REFRESH si:
    - target delta absente
    - Postgres vide mais Delta non vide
    - Delta contient au moins 1 id absent de Postgres (=> plus un miroir)
    """
    # ids only from Postgres
    pg_ids = (spark.read.jdbc(
        DB_URL,
        f"(SELECT {pk_col} FROM {pg_table}) AS q",
        properties=DB_PROPS
    ).select(F.col(pk_col).cast("string").alias("id")))

    if not spark.catalog.tableExists(delta_table):
        print(f"[{pg_table}] delta table missing -> FULL REFRESH")
        return True

    dl_ids = spark.table(delta_table).select(F.col(pk_col).cast("string").alias("id"))

    pg_empty = pg_ids.limit(1).count() == 0
    dl_empty = dl_ids.limit(1).count() == 0
    if pg_empty and not dl_empty:
        print(f"[{pg_table}] Postgres empty but Delta not empty -> FULL REFRESH")
        return True

    # Delta has extra ids not in Postgres => not a mirror
    # left_anti: rows in dl_ids not matching pg_ids
    extra_exists = (dl_ids.join(F.broadcast(pg_ids), on="id", how="left_anti")
                    .limit(1)
                    .count() > 0)
    if extra_exists:
        print(f"[{pg_table}] Delta has ids not in Postgres -> FULL REFRESH")
        return True

    return False

def enrich_df(pg_table: str, df, fk_mappings: list):
    # audit
    df = df.withColumn("_ingested_at", F.current_timestamp())

    # ---- Enrich created_by / updated_by from public."user"
    user_cols = [c for c in ["created_by_user_id", "updated_by_user_id"] if c in df.columns]
    if user_cols:
        user_lk = read_user_lookup()

        # created_by
        if "created_by_user_id" in user_cols:
            cb = (user_lk
                .withColumnRenamed("lk_user_id", "lk_cb_uuid")
                .withColumnRenamed("lk_user_gid", "lk_cb_gid")
                .withColumnRenamed("lk_user_firstname", "lk_cb_firstname")
                .withColumnRenamed("lk_user_lastname", "lk_cb_lastname"))

            df = (df
                .join(F.broadcast(cb), df["created_by_user_id"] == F.col("lk_cb_uuid"), "left")
                .withColumn("created_by_user_id", F.coalesce(F.col("lk_cb_gid"), F.col("created_by_user_id")))
                .withColumn("created_by_firstname", F.col("lk_cb_firstname"))
                .withColumn("created_by_lastname",  F.col("lk_cb_lastname"))
                .drop("lk_cb_uuid", "lk_cb_gid", "lk_cb_firstname", "lk_cb_lastname")
            )

        # updated_by
        if "updated_by_user_id" in user_cols:
            ub = (user_lk
                .withColumnRenamed("lk_user_id", "lk_ub_uuid")
                .withColumnRenamed("lk_user_gid", "lk_ub_gid")
                .withColumnRenamed("lk_user_firstname", "lk_ub_firstname")
                .withColumnRenamed("lk_user_lastname", "lk_ub_lastname"))

            df = (df
                .join(F.broadcast(ub), df["updated_by_user_id"] == F.col("lk_ub_uuid"), "left")
                .withColumn("updated_by_user_id", F.coalesce(F.col("lk_ub_gid"), F.col("updated_by_user_id")))
                .withColumn("updated_by_firstname", F.col("lk_ub_firstname"))
                .withColumn("updated_by_lastname",  F.col("lk_ub_lastname"))
                .drop("lk_ub_uuid", "lk_ub_gid", "lk_ub_firstname", "lk_ub_lastname")
            )


    # ---- Enrich simple user_id (kpi_comment): UUID -> gid + names
    if "user_id" in df.columns:
        user_lk = read_user_lookup()

        u = (user_lk
             .withColumnRenamed("lk_user_id", "lk_uuid")
             .withColumnRenamed("lk_user_gid", "lk_gid")
             .withColumnRenamed("lk_user_firstname", "lk_firstname")
             .withColumnRenamed("lk_user_lastname", "lk_lastname"))

        df = (df
            .join(F.broadcast(u), df["user_id"] == F.col("lk_uuid"), "left")
            .withColumn("user_id", F.coalesce(F.col("lk_gid"), F.col("user_id")))  # uuid -> gid
            .withColumn("user_firstname", F.col("lk_firstname"))
            .withColumn("user_lastname",  F.col("lk_lastname"))
            .drop("lk_uuid", "lk_gid", "lk_firstname", "lk_lastname")
        )


    # exercise enrichment
    if pg_table == "public.kpi_data" and "exercise_id" in df.columns:
        ex_lk = read_exercise_enrich_lookup()
        df = (df
              .join(F.broadcast(ex_lk), df["exercise_id"] == ex_lk["lk_exercise_id"], "left")
              .withColumn("exercise_source_id", F.col("lk_exercise_source_id"))
              .withColumn("exercise_scenario_id", F.col("lk_scenario_id"))
              .withColumn("data_domain_source_id", F.col("lk_data_domain_source_id"))
              .drop("lk_exercise_id", "lk_exercise_source_id", "lk_scenario_id", "lk_data_domain_source_id"))
        
    # ---- For kpi_log: rename user_firstname/lastname -> firstname/name
    if pg_table == "public.kpi_log":
        if "user_firstname" in df.columns:
            df = df.withColumnRenamed("user_firstname", "firstname")
        if "user_lastname" in df.columns:
            df = df.withColumnRenamed("user_lastname", "name")


    # ---- Enrich kpi_log via kpi_data_id (ONLY kpi_log)
    if pg_table == "public.kpi_log" and "kpi_data_id" in df.columns:

        kd_lk = read_kpi_data_enrich_lookup()
        kc_lk = read_kpi_comment_latest_lookup()

        # join kpi_data enrich
        df = (
            df
            .join(F.broadcast(kd_lk), df["kpi_data_id"] == kd_lk["lk_kpi_data_id"], "left")
            .withColumn("exercise_source_id",      F.col("lk_exercise_source_id"))
            .withColumn("exercise_scenario_id",    F.col("lk_exercise_scenario_id"))
            .withColumn("data_domain_source_id",   F.col("lk_data_domain_source_id"))
            .withColumn("signal_source_id",        F.col("lk_signal_source_id"))
            .withColumn("unit_source_id",          F.col("lk_unit_source_id"))
            .withColumn("version_source_id",       F.col("lk_version_source_id"))
            .withColumn("granularity",  F.col("lk_granularity"))
            .withColumn("period_year",  F.col("lk_period_year"))
            .withColumn("period_month", F.col("lk_period_month"))
            .drop(
                "lk_kpi_data_id",
                "lk_exercise_source_id",
                "lk_exercise_scenario_id",
                "lk_data_domain_source_id",
                "lk_signal_source_id",
                "lk_unit_source_id",
                "lk_version_source_id"
            )
        )

        # join dernier comment
        df = (
            df
            .join(F.broadcast(kc_lk), df["kpi_data_id"] == kc_lk["lk_kpi_data_id"], "left")
            .withColumn("comment", F.col("lk_comment"))
            .drop("lk_kpi_data_id", "lk_comment")
        )

    

    # FK UUID -> source_id
    for fk_col, lk_table, lk_pk, lk_source_col, out_col in fk_mappings:
        if fk_col not in df.columns:
            continue
        lk_df = read_lookup(lk_table, lk_pk, lk_source_col)
        df = (df
              .join(F.broadcast(lk_df), df[fk_col] == lk_df["lk_id"], "left")
              .withColumn(out_col, F.col("lk_source_id"))
              .drop("lk_id", "lk_source_id"))
        

    # remove all technical lookup columns before write/merge
    lk_cols = [c for c in df.columns if c.startswith("lk_")]
    if lk_cols:
        df = df.drop(*lk_cols)

    return df

def full_refresh(pg_table: str, delta_table: str, pk_col: str, created_col: str, updated_col: str, fk_mappings: list):
    print(f"[{pg_table}] Running FULL REFRESH -> overwrite {delta_table}")

    src_df = spark.read.jdbc(DB_URL, pg_table, properties=DB_PROPS)
    src_df = enrich_df(pg_table, src_df, fk_mappings)

    # overwrite target (mirror strict)
    (src_df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(qname(delta_table)))

    # recalage watermark sur le max event_time
    event_time = F.coalesce(F.col(updated_col), F.col(created_col), F.to_timestamp(F.lit("1970-01-01 00:00:00")))
    wm = (src_df
          .select(event_time.alias("et"), F.col(pk_col).cast("string").alias("pk"))
          .orderBy(F.col("et").desc(), F.col("pk").desc())
          .limit(1)
          .collect())
    if len(wm) == 0:
        # table vide
        set_watermark(pg_table, "1970-01-01 00:00:00", None)
        print(f"[{pg_table}] FULL REFRESH done (empty table) | watermark reset")
        return

    new_ts_str = wm[0]["et"].strftime("%Y-%m-%d %H:%M:%S")
    new_pk = wm[0]["pk"]
    set_watermark(pg_table, new_ts_str, new_pk)
    print(f"[{pg_table}] FULL REFRESH done | new watermark: ({new_ts_str}, {new_pk})")

# =========================
# MAIN LOOP
# =========================
for t in TABLES:
    pg_table = t["pg_table"]
    delta_table = t["delta_table"]
    pk_col = t["pk_col"]
    created_col = t["created_col"]
    updated_col = t["updated_col"]
    fk_mappings = t.get("fk_mappings", [])

    # --- Guardrail miroir strict
    if should_full_refresh(pg_table, delta_table, pk_col):
        full_refresh(pg_table, delta_table, pk_col, created_col, updated_col, fk_mappings)
        continue

    # --- Incremental path (ton watermark + merge)
    last_ts, last_pk = get_watermark(pg_table)

    pk_tie_clause = "TRUE" if last_pk is None else f"{pk_col} > '{last_pk}'"
    event_expr = f"COALESCE({updated_col}, {created_col}, TIMESTAMP '1970-01-01 00:00:00')"

    query = f"""
    (SELECT
        *,
        {event_expr} AS _event_time
     FROM {pg_table}
     WHERE
       ({event_expr} > TIMESTAMP '{last_ts}')
       OR
       ({event_expr} = TIMESTAMP '{last_ts}' AND {pk_tie_clause})
    ) AS q
    """

    inc_df = spark.read.jdbc(DB_URL, query, properties=DB_PROPS)

    if df_is_empty(inc_df):
        print(f"[{pg_table}] No changes since ({last_ts}, {last_pk})")
        continue

    inc_df = enrich_df(pg_table, inc_df, fk_mappings)

    # ensure target exists/schema
    ensure_delta_table_exists_sql(delta_table, inc_df.drop("_event_time"))
    ensure_delta_schema_has_columns(delta_table, inc_df.drop("_event_time"))

    # De-dupe within batch by pk
    inc_dedup = (inc_df
                 .orderBy(F.col("_event_time").desc(), F.col(pk_col).desc())
                 .dropDuplicates([pk_col])
                 .drop("_event_time"))

    # Merge
    merge_into(delta_table, inc_dedup, pk_col)

    # Update watermark
    last_row = (inc_df
                .select(F.col("_event_time").alias("et"), F.col(pk_col).cast("string").alias("pk"))
                .orderBy(F.col("et").desc(), F.col("pk").desc())
                .limit(1)
                .collect()[0])

    new_ts_str = last_row["et"].strftime("%Y-%m-%d %H:%M:%S")
    new_pk = last_row["pk"]
    set_watermark(pg_table, new_ts_str, new_pk)

    print(f"[{pg_table}] MERGED {inc_dedup.count()} rows into {delta_table} | new watermark: ({new_ts_str}, {new_pk})")
